In [1]:
import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [2]:
DATA_DIR = "/kaggle/input/datasets/ababeiraul/waicc2025"
train_df = pd.read_csv(f"{DATA_DIR}/train_data.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test_data.csv")

def clean_ro(series):
    return (
        series.fillna("").astype(str)
        .str.replace("ţ", "ț").str.replace("ş", "ș")
        .str.replace("Ţ", "Ț").str.replace("Ş", "Ș")
    )

for df in [train_df, test_df]:
    df["clean_word"] = clean_ro(df["word"])
    df["group_key"] = df["word_id"].astype(str).str.rsplit("_", n=1).str[0]
    df["word_idx"] = df["word_id"].astype(str).str.rsplit("_", n=1).str[1].astype(int)

In [3]:
def build_context_lookup(df_list, window=25):
    all_words = (
        pd.concat([d[["word_id", "group_key", "word_idx", "clean_word"]] for d in df_list])
        .drop_duplicates("word_id")
        .sort_values(["group_key", "word_idx"])
    )
    context_map = {}
    for group_key, grp in all_words.groupby("group_key"):
        words = grp["clean_word"].tolist()
        w_ids = grp["word_id"].tolist()
        n = len(words)
        for i in range(n):
            left = " ".join(words[max(0, i - window):i])
            right = " ".join(words[i + 1:min(n, i + 1 + window)])
            # Include text genre prefix + marked target word
            genre = group_key.split("_")[0]
            marked = f"[{genre}] {left} [TGT] {words[i]} [/TGT] {right}".strip()
            context_map[w_ids[i]] = marked
    return context_map

In [4]:
context_map = build_context_lookup([train_df, test_df], window=25)
train_df["input_text"] = train_df["word_id"].map(context_map)
test_df["input_text"] = test_df["word_id"].map(context_map)

In [5]:
print(train_df["input_text"].iloc[10])

[enc] Luna https://ro.wikipedia.org/wiki/Luna De la Wikipedia, enciclopedia liberă Luna este singurul [TGT] satelit [/TGT] natural al Pământului. Ea orbitează în jurul Pământului la o distanță medie de 384.400 km (238.900 mile; 1,28 secunde lumină), sau aproximativ de 30 de


In [6]:
gkf = GroupKFold(n_splits=5)
tr_idx, val_idx = next(gkf.split(train_df, train_df["answer"], groups=train_df["participant_id"]))

train_split = train_df.iloc[tr_idx].copy()
val_split = train_df.iloc[val_idx].copy()

# Aggregate training rows by word_id (faster training & removes participant-level noise)
train_agg = (
    train_split.groupby(["word_id", "input_text"], as_index=False)["answer"]
    .mean()
    .rename(columns={"answer": "label"})
)

# Keep validation rows un-aggregated so CV matches the exact leaderboard metric
val_eval = val_split[["word_id", "input_text", "answer"]].rename(columns={"answer": "label"})

print(f"Unique training words: {len(train_agg)} | Validation rows (unseen participants): {len(val_eval)}")

Unique training words: 4507 | Validation rows (unseen participants): 27042


In [7]:
MODEL_NAME = "dumitrescustefan/bert-base-romanian-cased-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Register target markers so BERT treats them as single atomic tokens
tokenizer.add_special_tokens({"additional_special_tokens": ["[TGT]", "[/TGT]"]})
TGT_TOKEN_ID = tokenizer.convert_tokens_to_ids("[TGT]")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1) # this is used in order to make the classifier a regressor
model.resize_token_embeddings(len(tokenizer))

def tokenize_fn(batch):
    return tokenizer(batch["input_text"], truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_agg[["input_text", "label"]]).map(tokenize_fn, batched=True, remove_columns=["input_text"])
val_ds = Dataset.from_pandas(val_eval[["input_text", "label"]]).map(tokenize_fn, batched=True, remove_columns=["input_text"])

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/397k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  500MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

Map:   0%|          | 0/4507 [00:00<?, ? examples/s]

Map:   0%|          | 0/27042 [00:00<?, ? examples/s]

In [9]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.clip(preds.squeeze(), 0, None)  # Reading time cannot be negative
    labels = labels.squeeze()

    r2 = max(0.0, r2_score(labels, preds, force_finite=True))
    pears = abs(pearsonr(labels, preds)[0])
    nitro_score = 100.0 * (r2 + pears) / 2.0

    return {
        "nitro_score": nitro_score,
        "r2": r2,
        "pearson": pears
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/robert_eyetracking",
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=0.1,                 
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="nitro_score",
    greater_is_better=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)


In [10]:
trainer.train()

# Save best model & tokenizer
trainer.save_model("/kaggle/working/best_robert_model")
tokenizer.save_pretrained("/kaggle/working/best_robert_model")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Nitro Score,R2,Pearson
1,No log,353796.531250,3.334771,0.000000,0.066695
2,No log,351086.906250,2.371104,0.000000,0.047422
3,No log,350061.937500,2.308981,0.000000,0.046180
4,No log,349755.843750,2.414036,0.000000,0.048281


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/best_robert_model/tokenizer_config.json',
 '/kaggle/working/best_robert_model/tokenizer.json')

In [ ]:
# Deduplicate test by word_id for fast inference
test_unique = test_df[["word_id", "input_text"]].drop_duplicates("word_id").reset_index(drop=True)
test_ds = Dataset.from_pandas(test_unique[["input_text"]]).map(tokenize_fn, batched=True, remove_columns=["input_text"])

raw_preds = trainer.predict(test_ds).predictions.squeeze()
test_unique["bert_pred"] = np.clip(raw_preds, 0, None)


In [ ]:
pred_map = test_unique.set_index("word_id")["bert_pred"]
submission = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test_df["datapointID"],
    "answer": np.round(test_df["word_id"].map(pred_map), 4)
})

submission.to_csv("/kaggle/working/submission_robert.csv", index=False)
print("Nulls:", submission.isna().sum().sum(), "| Shape:", submission.shape)
print(submission.head(10))